## Simple MultiAI Agent Architecture

In [2]:
import os
from typing import TypedDict, Annotated, List, Literal
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, END
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [4]:
from langgraph.graph import StateGraph, END, MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

In [7]:
class AgentState(MessagesState):
    next_agent: str #which agent should go next

In [8]:
#Create Simple tools
@tool
def search_web(query: str) -> str:
    """Search the web for informtion"""
    #Using Tavily for web search
    search = TavilySearchResults(max_results= 3)
    results = search.invoke(query)
    return str(results)

@tool
def write_summary(content: str) -> str:
    """Write a summary of the provuded content"""
    #Simple Summary Generation
    summary = f"summary of findings: \n\n{content[:500]}..."
    return summary

In [9]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("groq:llama-3.1-8b-instant")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A14F046F90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A14F4C4440>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [10]:
#Define agent functions (simpler approach)
def reasearcher_agent(state: AgentState):
    """Reasearcher agent that searches for information"""

    messages = state["messages"]

    #Add system message for context
    system_msg = SystemMessage(content="You are a reasearch assistant. Use the search_web tool  to find information about the user's request")

    #Call LLM with tools
    researcher_llm = llm.bind_tools([search_web])
    response = researcher_llm.invoke([system_msg] + messages)

    #Return the rsponse and route to writer
    return {
        "messages" : [response],
        "next_agent" : "writer"
    }

In [11]:
def writer_agent(state: AgentState):
    """Writer agent that creates summaries"""

    messages = state["messages"]

    #Add system message for context
    system_msg = SystemMessage(content="You are a technical writer. Review the conversation and create a clear, concise summary of the findings.")

    #Simple completion without tools
    response = llm.invoke([system_msg] + messages)

    return{
        "messages": [response],
        "next_agent": "end"
    }